# iTransformer trên PTB-XL: LiRA + RMIA (TV3, cls)

Chạy **sau** `itransformer_colab.ipynb`. Train K=8 shadow bằng đúng recipe target, rồi tính LiRA online/offline và RMIA offline/online.


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/Final_Project'
ITX_SRC = f'{DRIVE_ROOT}/itransformer'      # folder containing setup.py
DATASET_DIR = f'{DRIVE_ROOT}/dataset'
PTBXL_ZIP = f'{DATASET_DIR}/archive.zip'
RESULTS_DIR = f'{DRIVE_ROOT}/results/itransformer/cls'
CKPT_DIR = f'{DRIVE_ROOT}/ckpt/itransformer_cls'

!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
!rm -rf /content/itransformer && cp -r {ITX_SRC} /content/itransformer
!pip install -q /content/itransformer wfdb scikit-learn pandas tqdm matplotlib


## 1. Cấu hình


In [ ]:
SEED = 42
LEVEL = 'record'
EPOCHS = 50
BATCH = 16
MAX_LR = 1e-3
K = 8
N_VAL = 500
N_BOOT = 1000
RMIA_GAMMA, RMIA_A = 2.0, 0.3
CKPT_EVERY = 5
CKPT_DIR = f'{DRIVE_ROOT}/ckpt/itransformer_cls'
TARGET_EPOCH = 50
TAG = f'itx_{LEVEL}_s{SEED}'
D_MODEL, N_HEADS, E_LAYERS, D_FF, DROPOUT = 128, 8, 3, 256, 0.1
SEQ_LEN = 512
PTBXL_ROOT = '/content/ptbxl_raw'
SPLIT_PATH = f'{DATASET_DIR}/splits/ptbxl_{LEVEL}_s{SEED}.json'
CACHE_DIR = f'/content/cache/itx_cls_{LEVEL}_s{SEED}'
SHADOW_MASK = f'{DATASET_DIR}/splits/shadow_masks_record_K8_s42.npz'


## 2. Dữ liệu và target


In [ ]:
import os, glob
if not glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True):
    os.makedirs(PTBXL_ROOT, exist_ok=True)
    assert os.path.exists(PTBXL_ZIP), f'missing {PTBXL_ZIP}; {DATASET_DIR} has {os.listdir(DATASET_DIR)}'
    !unzip -q -o "{PTBXL_ZIP}" -d {PTBXL_ROOT}
found = glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True)
assert found, f'no ptbxl_database.csv after unzip; {PTBXL_ROOT} has {os.listdir(PTBXL_ROOT)}'
PTBXL_DIR = os.path.dirname(found[0])
print(PTBXL_DIR, os.listdir(PTBXL_DIR))


In [ ]:
import json, random, time, os
import numpy as np, torch, pandas as pd
from itransformer.ptbxl import PTBXLDataset, load_split_meta

def control_randomness(seed=SEED):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

split, meta = load_split_meta(SPLIT_PATH)
member_ds = PTBXLDataset(PTBXL_DIR, SPLIT_PATH, 'train', SEQ_LEN, CACHE_DIR)
val_ds    = PTBXLDataset(PTBXL_DIR, SPLIT_PATH, 'val',   SEQ_LEN, CACHE_DIR)
nonmem_ds = PTBXLDataset(PTBXL_DIR, SPLIT_PATH, 'test',  SEQ_LEN, CACHE_DIR)
pool_ds   = PTBXLDataset(PTBXL_DIR, SPLIT_PATH, 'pool',  SEQ_LEN, CACHE_DIR)
n_m, n_v, N = len(member_ds), len(val_ds), len(pool_ds)
pool_ids = pool_ds.ids; pool_y = pool_ds.labels
print('pool', N, 'member', n_m, 'val', n_v, 'nonmem', len(nonmem_ds))

tgt = torch.load(f'{CKPT_DIR}/{TAG}_logits_ep{TARGET_EPOCH}.pt')
TARGET_KEY = f"itx|cls|{LEVEL}|{SEED}|{EPOCHS}|{BATCH}|{MAX_LR}|{D_MODEL}|{meta['sha256'][:12]}"
mk = np.load(SHADOW_MASK)
order = np.argsort(mk['ids'])
assert len(mk['ids']) == N and (mk['ids'][order] == pool_ids).all()
masks_pool = mk['masks'][:K][:, order]
ref_pool = np.flatnonzero(np.isin(pool_ids, mk['ref_ids']))
print('K', K, 'in-fraction', masks_pool.mean().round(3), 'ref', len(ref_pool))


## 3. Train K shadow


In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
from itransformer.model import ITransformer

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
amp = lambda: torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=bf16)

def build_model(name):
    model = ITransformer('classification', SEQ_LEN, n_variates=12, n_class=5,
                         d_model=D_MODEL, n_heads=N_HEADS, e_layers=E_LAYERS, d_ff=D_FF, dropout=DROPOUT)
    print(f'[{name}] {sum(p.numel() for p in model.parameters())/1e6:.2f}M')
    return model.to(dev)

@torch.no_grad()
def get_logits(model, ds, bs=64):
    model.eval(); out = []
    for x, _ in DataLoader(ds, batch_size=bs, shuffle=False):
        with amp(): out.append(model(x.to(dev).float()).float().cpu())
    return torch.cat(out)

def save_atomic(obj, path):
    tmp = path + '.tmp'; torch.save(obj, tmp); os.replace(tmp, path)
    with open(path, 'rb') as _f: os.fsync(_f.fileno())

def train_shadow(j):
    tag = f'shadow_{LEVEL}_s{SEED}_k{j}'
    out_path, ckpt_path = f'{CKPT_DIR}/{tag}_logits.pt', f'{CKPT_DIR}/{tag}.pt'
    key = f"{TARGET_KEY}|shadow{j}|{meta['shadow']['sha256'][:12]}"
    if os.path.exists(out_path) and torch.load(out_path).get('key') == key:
        print(f'[shadow {j}] skip'); return
    control_randomness(SEED + 1 + j)
    model = build_model(f'shadow {j}')
    idx = np.flatnonzero(masks_pool[j])
    loader = DataLoader(Subset(pool_ds, idx), batch_size=BATCH, shuffle=True, drop_last=True)
    params = list(model.parameters())
    opt = torch.optim.AdamW(params, lr=MAX_LR / 25)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=MAX_LR, total_steps=EPOCHS * len(loader))
    ce = torch.nn.CrossEntropyLoss(); start = 1
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location='cpu')
        if ck.get('key') == key:
            model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
            start = ck['epoch'] + 1
        del ck
    for ep in range(start, EPOCHS + 1):
        model.train(); losses = []
        for x, yb in tqdm(loader, desc=f'shadow {j} ep{ep}', leave=False):
            opt.zero_grad(set_to_none=True)
            with amp(): logits = model(x.to(dev).float())
            loss = ce(logits.float(), yb.to(dev)); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step(); losses.append(loss.item())
        print(f'[shadow {j}] ep{ep} loss {np.mean(losses):.4f}')
        if ep % CKPT_EVERY == 0 and ep < EPOCHS:
            save_atomic({'key': key, 'epoch': ep, 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                         'model': {k: v.cpu() for k, v in model.state_dict().items()}}, ckpt_path)
    lg = get_logits(model, pool_ds)
    hit = (lg.argmax(1).numpy() == pool_y); m = masks_pool[j]
    print(f'[shadow {j}] acc in {hit[m].mean():.4f} out {hit[~m].mean():.4f}')
    save_atomic({'key': key, 'logits': lg}, out_path)
    if os.path.exists(ckpt_path): os.remove(ckpt_path)
    del model, opt
    if torch.cuda.is_available(): torch.cuda.empty_cache()

for j in range(K):
    t0 = time.time(); train_shadow(j); print(f'shadow {j} done in {(time.time()-t0)/60:.1f} min')


## 4. LiRA và RMIA


In [ ]:
from scipy.stats import norm
from sklearn.metrics import roc_curve, auc as sk_auc
from itransformer.mia import mia_metrics, bootstrap_ci

S = torch.stack([torch.load(f'{CKPT_DIR}/shadow_{LEVEL}_s{SEED}_k{j}_logits.pt')['logits'] for j in range(K)])

def phi_p(logits, y):
    z = logits.double(); y = torch.as_tensor(y).long()[:, None]
    phi = z.gather(1, y).squeeze(1) - torch.logsumexp(z.scatter(1, y, float('-inf')), 1)
    return phi.numpy(), torch.sigmoid(phi).numpy()

PHI_S, P_S = map(np.stack, zip(*[phi_p(S[j], pool_y) for j in range(K)]))
T = torch.full((N, 5), float('nan'))
idx_m, idx_n = np.arange(n_m), np.arange(n_m + n_v, N)
T[idx_m], T[idx_n] = tgt['lm'], tgt['ln']
PHI_T, P_T = phi_p(T, pool_y)
idx_t = np.r_[idx_m, idx_n]
Min = masks_pool.astype(float); Mout = 1 - Min

def group_stats(W, X):
    c = W.sum(0); mu = (W * X).sum(0) / c
    return mu, (W * (X - mu) ** 2).sum(0) / (c - 1)

def lira(online):
    mu_in, v_in = group_stats(Min, PHI_S); mu_out, v_out = group_stats(Mout, PHI_S)
    s_in, s_out = np.sqrt(v_in[idx_t].mean()), np.sqrt(v_out[idx_t].mean())
    x = PHI_T[idx_t]
    if online: return norm.logpdf(x, mu_in[idx_t], s_in) - norm.logpdf(x, mu_out[idx_t], s_out)
    return (x - mu_out[idx_t]) / s_out

def rmia(online):
    p_in = (Min * P_S).sum(0) / Min.sum(0); p_out = (Mout * P_S).sum(0) / Mout.sum(0)
    p_bar = 0.5 * (p_in + p_out) if online else 0.5 * ((1 + RMIA_A) * p_out + (1 - RMIA_A))
    ratio = P_T / p_bar
    z = ratio[np.intersect1d(ref_pool, idx_t)]
    return np.searchsorted(np.sort(z), ratio[idx_t] / RMIA_GAMMA, side='right') / len(z)

scores = {'lira': lira(True), 'lira_offline': lira(False), 'rmia': rmia(False), 'rmia_online': rmia(True)}
print({k: v.shape for k, v in scores.items()})

# utility from target logits
acc_m = float((tgt['lm'].argmax(1).numpy() == member_ds.labels).mean())
acc_n = float((tgt['ln'].argmax(1).numpy() == nonmem_ds.labels).mean())
os.makedirs(RESULTS_DIR, exist_ok=True)
rows = []
for a, s in scores.items():
    sm, sn = s[:n_m], s[n_m:]
    t01, t1, au = mia_metrics(sm, sn); c01, c1 = bootstrap_ci(sm, sn, N_BOOT, SEED)
    out = {
        'model': 'itransformer', 'task': 'cls', 'attack': a, 'owner_tv': 3,
        'level': LEVEL, 'dataset': 'ptbxl', 'seed': SEED,
        'n_member': int(len(sm)), 'n_nonmember': int(len(sn)),
        'scores_member': np.round(sm, 6).tolist(), 'scores_nonmember': np.round(sn, 6).tolist(),
        'tpr_at_0.1fpr': t01, 'tpr_at_1fpr': t1, 'auc': au,
        'bootstrap_ci': {'n_resamples': N_BOOT, 'confidence': 0.95, 'tpr_at_0.1fpr': c01, 'tpr_at_1fpr': c1},
        'utility': {'train_acc': acc_m, 'test_acc': acc_n, 'gap': acc_m - acc_n},
        'n_shadow': K,
        'extra': {'epochs': EPOCHS, 'eval_epoch': TARGET_EPOCH, 'max_lr': MAX_LR, 'batch': BATCH,
                  'backbone': 'iTransformer', 'gamma': RMIA_GAMMA, 'a': RMIA_A},
    }
    json.dump(out, open(f'{RESULTS_DIR}/{a}.json', 'w'))
    rows.append({'attack': a, 'tpr@0.1': t01, 'tpr@1': t1, 'auc': au})
    print('saved', a)
pd.DataFrame(rows)
